# Notebook 5 of 5 · Safety Evaluation & Governance Framework
### Assembling the KRI dashboard — KB Bank accountability applied to an LLM control

> This notebook turns the metrics from NB2–4 into a **governed control**: severity-tiered
> thresholds, a per-identity-group fairness KRI, a red-team release gate, and statistical
> drift alerting with RAG escalation — the KB Bank executive-accountability / KB Life KRI
> discipline applied to LLM safety. Framework: [docs/LLM_SAFETY_FRAMEWORK.md](../docs/LLM_SAFETY_FRAMEWORK.md).

In [ ]:
# --- repo bootstrap: make `from src import ...` work from notebooks/ ---
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titleweight": "bold", "font.size": 10})

import src
from src import data_loader as dl
from src.data_loader import LABELS, ANTHROPIC_POLICY_MAP, SEVERITY_ORDER
print("src loaded. LABELS:", LABELS)

In [ ]:
from src import ml_baseline as mlb
from src import safety_metrics as sm
from src import red_team_generator as rt

df = dl.load_jigsaw()
try:
    pipe = mlb.load_model()
    _, splits = mlb.train_baseline(df)            # to get a held-out test split
except Exception:
    pipe, splits = mlb.train_baseline(df)
proba, pred = mlb.predict(pipe, splits["X_test"])
y_true = splits["Y_test"].values
print("Control loaded; evaluating on", len(splits["X_test"]), "held-out comments.")

## 1 · Severity-tiered thresholds (KB-Life-KRI logic)

> A single 0.5 cut treats a missed `threat` like a missed `obscene`. We apply
> per-category thresholds set by severity tier and quantify the recall lift on the harms
> that matter most.

In [ ]:
print("Tiered thresholds:", sm.tiered_thresholds())
pred_flat  = (proba.values >= 0.5).astype(int)
pred_tier  = sm.apply_tiered_thresholds(proba.values)
flat = sm.per_category_metrics(y_true, pred_flat).set_index("category")
tier = sm.per_category_metrics(y_true, pred_tier).set_index("category")
lift = pd.DataFrame({
    "severity":[ANTHROPIC_POLICY_MAP[l]["severity_tier"] for l in LABELS],
    "recall_flat": flat.loc[LABELS,"recall"].values,
    "recall_tiered": tier.loc[LABELS,"recall"].values,
    "precision_tiered": tier.loc[LABELS,"precision"].values,
}, index=LABELS)
lift["recall_lift"] = (lift["recall_tiered"]-lift["recall_flat"]).round(3)
lift.round(3)

## 2 · Fairness KRI (K2) — per-identity-group false-positive rate

> The LLM analogue of fair-lending disparate-impact monitoring (Project 2). A *clean*
> comment that mentions an identity group and gets flagged is a false positive that
> silences the very group the control should protect.

In [ ]:
fpr = sm.group_false_positive_rates(splits["X_test"].values,
                                    splits["Y_test"]["toxic"].values,
                                    pred_tier[:, LABELS.index("toxic")],
                                    dl.IDENTITY_TERMS, label="toxic")
overall = fpr.attrs["overall_fpr_pct"]
fpr["gap_vs_overall_pp"] = (fpr["false_positive_rate_pct"] - overall).round(2)
print(f"Overall toxic FPR: {overall:.2f}%  |  K2 tolerance: 3.0 pp gap")
fpr["K2_status"] = np.where(fpr["gap_vs_overall_pp"] > 3.0, "RED — investigate",
                     np.where(fpr["gap_vs_overall_pp"] > 1.5, "AMBER — watch", "GREEN"))
fpr.round(2)

## 3 · Drift KRI (K1) — statistical alerting with RAG escalation

> We simulate a monitoring cycle: take the NB2 scorecard as the **baseline**, and a
> degraded **current** run (e.g. data drift erodes recall), then let `drift_kri` flag
> categories whose recall has dropped beyond the **severity-tiered tolerance**. A Critical
> category trips RED on a far smaller drop than a Medium one — the consequence-scaled
> threshold that is the heart of the KB-Life-KRI idea.

In [ ]:
baseline_sc = sm.per_category_metrics(y_true, pred_tier)
# simulate a degraded current cycle: knock recall down unevenly (drift)
rng = np.random.RandomState(7)
current_sc = baseline_sc.copy()
current_sc["recall"] = (current_sc["recall"] * rng.uniform(0.85, 0.99, len(current_sc))).clip(0, 1)
kri = sm.drift_kri(baseline_sc, current_sc, metric="recall")
kri

In [ ]:
# RAG dashboard visual
colmap = {"GREEN — ok":"#55A868","AMBER — watch":"#DD8452","RED — escalate":"#C44E52"}
fig, ax = plt.subplots(figsize=(10, 3.2))
ax.bar(kri["category"], kri["drop"], color=[colmap[s] for s in kri["kri_status"]])
for i, t in enumerate(kri["tolerance"]):
    ax.hlines(t, i-0.4, i+0.4, color="black", lw=1.2, linestyles="--")
ax.set_title("K1 drift KRI — recall drop vs severity-tiered tolerance (dashed)")
ax.set_ylabel("recall drop"); ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()
print("Dashed line = tolerance for that category's severity tier. Bars above it escalate.")

## 4 · Red-team gate (K3)

> Pull the red-team scorecard from NB4 (recomputed here if its output isn't on disk) into
> the KRI dashboard. Benign-identity must be 100%; overall ≥ 90%.

In [ ]:
try:
    ml_scored = pd.read_csv(os.path.join(dl.outputs_dir(), "red_team_ml_scored.csv"))
except Exception:
    from src.llm_client import ToxicityClassifier, assessments_to_frame
    cases = rt.generate_cases()
    _, mlp = mlb.predict(pipe, cases["text"])
    ml_scored = rt.score_predictions(cases, mlp)
k3_overall = 100 * ml_scored["outcome_correct"].mean()
k3_benign = rt.suite_summary(ml_scored).set_index("attack_suite").loc[
    "benign_identity_false_positive", "pass_rate_pct"]
print(f"K3 red-team — overall pass {k3_overall:.1f}% (gate 90%) | "
      f"benign-identity {k3_benign:.1f}% (gate 100%)")

## 5 · The governance dashboard — one view, RAG-coded

> The executive-accountability view: every KRI, its status, and its owner. A RED on a
> Critical category is treated as a potential safety/discrimination incident, escalated to
> the Accountable Executive, not a routine metric miss.

In [ ]:
dashboard = pd.DataFrame([
    {"KRI":"K1 detection recall (Critical)", "metric":"min Critical recall drop",
     "status": "RED" if (kri[kri.severity_tier=="Critical"]["kri_status"].str.startswith("RED")).any() else "GREEN",
     "owner":"Control Owner → Exec"},
    {"KRI":"K2 identity FPR gap", "metric":f"max gap {fpr['gap_vs_overall_pp'].max():.1f}pp",
     "status": "RED" if (fpr["gap_vs_overall_pp"]>3).any() else ("AMBER" if (fpr['gap_vs_overall_pp']>1.5).any() else "GREEN"),
     "owner":"Model Risk Lead"},
    {"KRI":"K3 red-team pass-rate", "metric":f"overall {k3_overall:.0f}% / benign {k3_benign:.0f}%",
     "status": "PASS" if (k3_overall>=90 and k3_benign>=100) else "RED",
     "owner":"Model Risk Lead"},
    {"KRI":"K4 LLM reliability", "metric":"schema/hallucination (NB3 design)",
     "status":"GREEN (structured output enforced)", "owner":"Control Owner"},
])
dashboard.to_csv(os.path.join(dl.outputs_dir(), "governance_kri_dashboard.csv"), index=False)
dashboard

## 6 · Closing — what this project demonstrates

| Layer | Delivered |
|---|---|
| **Harm characterisation** | NB1: distribution, co-occurrence, bias scan |
| **Champion control** | NB2: interpretable, threshold-governable TF-IDF+LogReg |
| **Challenger control** | NB3: policy-native Claude classifier, routing trade-off |
| **Adversarial robustness** | NB4: version-controlled red-team battery, release gates |
| **Governance** | NB5: tiered thresholds, fairness KRI, drift KRI, RAG escalation |
| **Policy & regulation** | Anthropic Usage Policy mapping + EU AI Act (Art. 5,10,15,51–55) |

**The thesis, restated:** the same governance discipline that controls a bank's credit and
fraud models (thresholds, fairness monitoring, statistically-thresholded KRIs,
champion/challenger validation) carries all the way to an LLM safety control — *plus* the
LLM-specific layers (Usage-Policy mapping, prompt-as-control, red-teaming, hallucination
monitoring) that the frontier risk surface demands.

*See [docs/PORTFOLIO_INTEGRATION.md](../docs/PORTFOLIO_INTEGRATION.md) for how Projects 1–3 connect.*